# SECTION 08: Inventory Analysis

## Objective

In this section, we will analyze product inventory levels, stock availability, and warehouse distribution to identify inventory trends, optimize stock management, and support operational decision-making.

In [14]:
--Before going to perform the analysis, make sure to select the database in which you want to run the query.
/*Required tables:
    Production.Product
    production.productinventory
    Production.Location
*/

Commands completed successfully.

Total execution time: 00:00:00.004

In [ ]:
--Query.1

print 'Products with Low Stock';

with lowstocks as
(
    select p.productid,
        p.name,
        SUM(pi.quantity) as stockcount
    from production.product p
    join production.productinventory pi
        on p.productid = pi.productid
    group by p.productid, p.name
)

select *from lowstocks
order by stockcount asc
go

--Querry.2

print 'invenory Distribution by Location';

with location_distribution as
(
    select l.locationid,
        l.name as locationname,
        SUM(pi.quantity) as stockcount,
        count(distinct pi.productid) as productcount
    from production.location l    
    join production.productinventory pi
        on l.locationid = pi.locationid
    group by l.locationid, l.name
)
select *from location_distribution
order by stockcount desc;


--Querry.3

print 'Hightest stock product at eaach location';


with Hightest_product as

(
    select l.name as locationname,
        p.name as productname,
        SUM(pi.quantity) as stockcount
        from production.location l
        join production.productinventory pi
            on l.locationid = pi.locationid
        join production.product p on p.productid = pi.productid
        group by l.name, p.name
),
ranked_products as (
    select *,dense_rank() over(partition by locationname order by stockcount desc) as rank  from Hightest_product
)

select *from ranked_products
where rank = 1



# SECTION 09: Executive Dashboard

## Objective

In this section, we will build a single comprehensive SQL query that combines key business KPIs such as total revenue, total orders, total customers, average order value, best-selling product, and best-performing territory into one executive-level dashboard result.

In [ ]:
--Before going to perform the analysis, make sure to select the database in which you want to run the query.
/*Required tables:
    sales.salesorderheader	
    sales.salesorderdetail	
    production.product	
    sales.salesterritory
*/

In [16]:

--Key Performance Indicators (KPIs) 
--What is the most  effective product and territory in terms of revenue generation?
--This will give us an idea about which product and territory is generating the most revenue for the company.

with kpis as
(
    select
        sum(totaldue) as total_revenue,
        count(salesorderid) as total_orders,
        count(distinct customerid) as total_customers,
        avg(totaldue) as average_order_value
    from sales.salesorderheader
),

best_product as
(
    select top 1
        p.name as best_selling_product
    from sales.salesorderdetail s
    join production.product p
        on s.productid = p.productid
    group by
        p.productid,
        p.name
    order by sum(s.linetotal) desc
),

best_territory as
(
    select top 1
        t.name as best_territory
    from sales.salesorderheader h
    join sales.salesterritory t
        on h.territoryid = t.territoryid
    group by
        t.territoryid,
        t.name
    order by sum(h.totaldue) desc
)

select
    o.total_revenue,
    o.total_orders,
    o.total_customers,
    o.average_order_value,
    p.best_selling_product,
    t.best_territory
from kpis o
cross join best_product p
cross join best_territory t;
go

(1 row affected)

total_revenue  | total_orders | total_customers | average_order_value | best_selling_product   | best_territory
---------------+--------------+-----------------+---------------------+------------------------+---------------
123216786.1159 | 31465        | 19119           | 3915.9951           | Mountain-200 Black, 38 | Southwest     
(1 row)

Total execution time: 00:00:00.325

# SECTION 10: Advanced SQL Concepts

## Objective

In this section, we will apply advanced SQL features such as CTEs, Views, Stored Procedures, Scalar Functions, Inline Table-Valued Functions, and Temporary Tables to create reusable and business-oriented analytical solutions.

In [ ]:
--Query.1

/*The management team wants to identify customers whose total spending is above the average customer spending.*/


with hightest_spending_customers as
(
    select
        c.customerid,
        p.firstname +' '+ p.lastname as customername,
        sum(c.totaldue) as total_spending,
        avg(c.totaldue) as average_spending
    from sales.salesorderheader c
    join sales.person p
        on c.customerid = p.businessentityid
    group by 
        c.customerid,p.firstname +' '+ p.lastname
)
select *from hightest_spending_customers
where total_spending > average_spending
        